# 01 Earth polar DSM cleaning and preprocessing — executed verification notebook

The full source transformation, masked products, terrain metrics and 72 cast-shadow scenarios were **executed in** `code/run_earth_polar.py` (included in this delivery). This notebook is a lightweight executed verification and reusable entry point; it intentionally does not duplicate expensive preprocessing.

Run `python code/run_earth_polar.py` after setting its input and output paths to execute all processing again from unchanged source TIFFs. Read `README.md` before interpreting results.

In [1]:
from pathlib import Path
import json, csv, rasterio, numpy as np
ROOT=Path.cwd()
print('Processing directory:',ROOT.name)
print('Processing scripts:',len(list((ROOT/'code').glob('*.py'))))

Processing directory: Earth_Polar_Processed
Processing scripts: 1


In [2]:
sites=['Haughton_tile','Antarctic_tile','North_Greenland']
for s in sites:
    q=json.loads((ROOT/'reports'/f'{s}_quality_and_stats.json').read_text())
    print(s,'input finite pixels:',q['streamed_quality']['finite_pixels'],'core valid:',round(q['analysis']['core_valid_pct'],2),'%','scenarios:',len(q['shadow_scenarios']))
    assert len(q['shadow_scenarios'])==24

Haughton_tile input finite pixels: 4320000 core valid: 100.0 % scenarios: 24
Antarctic_tile input finite pixels: 4320000 core valid: 100.0 % scenarios: 24
North_Greenland input finite pixels: 2592000 core valid: 100.0 % scenarios: 24


In [3]:
for s in sites:
    with rasterio.open(ROOT/'processed/common'/f'{s}_core_dem_80m.tif') as ref:
        assert ref.res==(80.0,80.0) and ref.shape==(80,80)
        with rasterio.open(ROOT/'processed/terrain'/f'{s}_slope_deg.tif') as slope:
            assert slope.transform.almost_equals(ref.transform) and slope.crs==ref.crs
    with rasterio.open(ROOT/'processed/illumination'/f'{s}_castshadow_alt1p0_az000.tif') as m:
        assert set(np.unique(m.read(1))).issubset({0,1,2})
print('Validated all site output grids and shadow-mask code range')

Validated all site output grids and shadow-mask code range


In [4]:
d=json.loads((ROOT/'frontend'/'dashboard_data.json').read_text());g=json.loads((ROOT/'frontend'/'sites_wgs84.geojson').read_text())
assert len(d['sites'])==len(g['features'])==3
for site in d['sites']:
    for k in ['terrain_preview','core_dem','shadow_scenarios_csv']:
        assert (ROOT/'frontend'/site[k]).resolve().exists()
print('Frontend JSON, WGS84 GeoJSON and relative paths validated')

Frontend JSON, WGS84 GeoJSON and relative paths validated


In [5]:
tests=json.loads((ROOT/'reports'/'automated_tests.json').read_text())
assert all(tests.values())
print('Synthetic scenario tests passed:',', '.join(tests.keys()))

Synthetic scenario tests passed: flat_shadow_all_clear, obstacle_blocks_southward_solar_ray, gap_unresolved, edge_unresolved, known_plane_slope_deg, known_plane_aspect_east_deg
